In [2]:
!pip install xarray netCDF4 pandas

In [ ]:
# --- Step 1: Install necessary libraries (if you haven't already) ---
# This step only needs to be run once in your environment (e.g., Anaconda Jupyter).
# If using Google Colab, prepend with '!' like !pip install xarray netCDF4 pandas
# If in a regular terminal for Anaconda: conda install xarray netCDF4 pandas
# pip install xarray netCDF4 pandas

# --- Step 2: Import the required libraries ---
import xarray as xr
import pandas as pd
import os # For path manipulation if needed

# --- Step 3: Define the path to your downloaded NetCDF file ---
# Replace 'path/to/your/ceres_syn_data.nc' with the actual path to your file.
# If the file is in the same directory as this notebook, just use its filename:
# file_path = 'CERES_SYN1deg-1H_Ed4.1_YOUR_DATE_LA.nc'
# For example:
file_path = 'path/to/your/CERES_SYN1deg-1H_Ed4.1_20230101-20230107.nc'

# If you downloaded it to a specific folder, use os.path.join or a full path
# Example if in a 'data' subfolder:
# file_path = os.path.join('data', 'CERES_SYN1deg-1H_Ed4.1_20230101-20230107.nc')

# --- Step 4: Open the NetCDF file using xarray ---
# This loads the data into an xarray.Dataset object.
# It doesn't load all data into memory immediately, but provides an interface to it.
try:
    ds = xr.open_dataset(file_path)
    print(f"Successfully loaded dataset from: {file_path}")
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
    print("Please ensure the file path is correct and the file exists.")
    ds = None # Set ds to None to avoid errors if the file wasn't loaded
except Exception as e:
    print(f"An error occurred while opening the dataset: {e}")
    ds = None

# Proceed only if the dataset was successfully loaded
if ds is not None:
    # --- Step 5: Inspect the dataset structure ---
    # This is crucial! It will show you all variables, their dimensions (time, lat, lon),
    # attributes (units, long_name), and global attributes.
    print("\n--- Dataset Information (ds) ---")
    print(ds)

    # You can also get a quicker look at just the variables
    print("\n--- Dataset Variables (ds.data_vars) ---")
    print(ds.data_vars)

    # --- Step 6: Identify and access your desired variables ---
    # Based on our previous discussion, you're interested in:
    # 1. All-Sky TOA Spectral Longwave Up Flux
    # 2. Clear-Sky Profile Longwave Up Flux (at TOA or a specific pressure level)

    # IMPORTANT: The exact variable names in your downloaded .nc file might vary slightly.
    # Use the output from Step 5 (ds.data_vars) to find the precise names.
    # Common conventions for these might be:
    # 'toa_lw_all_mon', 'toa_lw_clr_mon', 'adj_allsky_TOA_longwave_up_flux', etc.

    # Placeholder variable names - replace with actual names from your ds.data_vars output
    all_sky_lw_flux_name = 'toa_lw_all_1h' # Example: check your file's ds.data_vars
    clear_sky_lw_flux_name = 'toa_lw_clr_1h' # Example: check your file's ds.data_vars

    if all_sky_lw_flux_name in ds and clear_sky_lw_flux_name in ds:
        all_sky_flux = ds[all_sky_lw_flux_name]
        clear_sky_flux = ds[clear_sky_lw_flux_name]

        print(f"\n--- Variable: {all_sky_lw_flux_name} ---")
        print(all_sky_flux)
        print(f"\n--- Variable: {clear_sky_lw_flux_name} ---")
        print(clear_sky_flux)

        # --- Step 7: Select data for your specific Los Angeles region and time ---
        # Assuming your data has 'time', 'lat', 'lon' dimensions
        
        # Define the bounding box for Los Angeles
        # Use a slightly wider range to ensure all relevant grid cells are caught
        lat_min, lat_max = 33.20, 35.00
        lon_min, lon_max = -119.50, -117.00

        # Select the spatial subset using .sel() or .sel(method='nearest')
        # Using .sel() with slices for ranges
        la_area_all_sky = all_sky_flux.sel(
            lat=slice(lat_min, lat_max),
            lon=slice(lon_min, lon_max)
        )
        la_area_clear_sky = clear_sky_flux.sel(
            lat=slice(lat_min, lat_max),
            lon=slice(lon_min, lon_max)
        )

        print("\n--- All-Sky Flux for Los Angeles Area (Subset) ---")
        print(la_area_all_sky)

        # --- Step 8: Convert to Pandas DataFrame for easier iteration/streaming ---
        # This will convert the xarray.DataArray (your selected flux) into a pandas DataFrame.
        # It stacks all dimensions into rows, which is ideal for a time series database.
        
        # Merge the two selected variables into a single Dataset for easier DataFrame conversion
        la_combined_data = xr.Dataset({
            'all_sky_lw': la_area_all_sky,
            'clear_sky_lw': la_area_clear_sky
        })

        # Convert the combined xarray Dataset to a Pandas DataFrame
        # .stack() is often useful if you have multiple spatial dimensions you want to flatten
        # .reset_index() turns the multi-index into columns
        df_la = la_combined_data.to_dataframe().reset_index()

        print("\n--- Sample of Los Angeles Data in Pandas DataFrame ---")
        print(df_la.head())
        print(f"\nTotal rows in LA DataFrame: {len(df_la)}")

        # --- Step 9: Prepare data for your streaming iterator ---
        # You can now iterate through this DataFrame, taking each row and formatting
        # it for your time series database ingestion.
        # Remember to convert timestamps to UTC if your database or Sift client requires it.
        # The 'time' column in df_la is already a datetime object.

        # Example of iterating and preparing a dictionary
        # from datetime import timezone
        # la_tz = ZoneInfo("America/Los_Angeles") # Assuming your data 'time' is in LA timezone

        # for index, row in df_la.iterrows():
        #     # Convert naive datetime from NetCDF to UTC if it's not already
        #     # (NetCDF datetimes can be naive or aware, check ds['time'].encoding for 'units')
        #     # If the 'time' coordinate in xarray is already timezone-aware (e.g., in UTC),
        #     # you might not need the localization step.
        #     if row['time'].tzinfo is None:
        #         aware_dt = row['time'].replace(tzinfo=la_tz) # Localize if naive
        #         utc_dt = aware_dt.astimezone(timezone.utc)
        #     else:
        #         utc_dt = row['time'].astimezone(timezone.utc) # Convert if already aware
            
        #     data_point_for_db = {
        #         'timestamp': utc_dt,
        #         'latitude': row['lat'],
        #         'longitude': row['lon'],
        #         'all_sky_lw': row['all_sky_lw'],
        #         'clear_sky_lw': row['clear_sky_lw']
        #     }
        #     print(data_point_for_db)
        #     # ingestion_service.ingest_flows(...)
        #     # Add a small sleep here if simulating streaming from a file

    else:
        print(f"\nCould not find expected variables: '{all_sky_lw_flux_name}' or '{clear_sky_lw_flux_name}' in the dataset.")
        print("Please check the `ds.data_vars` output above and update the variable names in the script.")

In [3]:
!pip install sift-stack-py

In [4]:
import os
 
from datetime import datetime, timezone
from random import random
from sift_py.grpc.transport import SiftChannelConfig, use_sift_channel
from sift_py.ingestion.channel import ChannelConfig, ChannelDataType, double_value, float_value
from sift_py.ingestion.config.telemetry import TelemetryConfig
from sift_py.ingestion.flow import FlowConfig
from sift_py.ingestion.service import IngestionService
from time import sleep, time
from typing import Iterator, Tuple

In [5]:
# mocks stream using a CSV of Denver Airport weather readings
import csv
from typing import Iterator, Dict, Any
from datetime import datetime
import time
from zoneinfo import ZoneInfo

def meteorological_data_source(file_path: str) -> Iterator[Dict[str, Any]]:
    """
    Reads a CSV file and yields a dictionary for each row,
    simulating a stream of meteorological data.

    Args:
        file_path: The path to the CSV file.

    Yields:
        A dictionary containing the date, temperature, dew point,
        wind speed, and precipitation for each record.
    """
    # Define the list of columns we want to extract
    column_names = ['DATE', 'TEMP', 'DEWP', 'MXSPD', 'PRCP']
    local_tz = ZoneInfo("America/Denver")

    try:
        with open(file_path, mode='r', encoding='utf-8') as csvfile:
            # Use a DictReader to automatically map rows to dictionaries
            reader = csv.DictReader(csvfile)

            # Check if all required columns are present in the CSV
            missing_columns = [col for col in column_names if col not in reader.fieldnames]
            if missing_columns:
                print(f"Error: The following columns are missing from the CSV: {missing_columns}")
                return

            # Simulate streaming by yielding one row at a time
            for row in reader:
                # Extract the desired data from the row
                naive_dt = datetime.strptime(row['DATE'], '%Y-%m-%d')
                aware_dt_la = naive_dt.replace(tzinfo=local_tz)
                data = {
                    'timestamp': aware_dt_la.astimezone(timezone.utc),
                    'temp': float(row['TEMP']),
                    'dewp': float(row['DEWP']),
                    'wdsp': float(row['MXSPD']),
                    'prcp': float(row['PRCP'])
                }

                yield data

    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found. Please check the path.")
    except (ValueError, TypeError) as e:
        print(f"Error processing data. Check for malformed values: {e}")

file_path = os.path.expanduser('~/developer/time_series.csv')

# 2. Use the iterator to process the data.


# print(data_source)
# for row in data_source:
#     print(row)

In [6]:
# Configure Ingestion for two datasets

# The name of our asset
ASSET_NAME = "CERES_longwave_up_flux"
 
# An arbitrary key used to identify our config
CONFIG_KEY = "CERES_longwave_up_flux-config-v1"
 
# Define the schema of your telemetry
config = TelemetryConfig(
    asset_name=ASSET_NAME,
    ingestion_client_key=CONFIG_KEY,
    flows=[
        FlowConfig(
            name="denver_weather",
            channels=[
                ChannelConfig(
                    name="temp",
                    data_type=ChannelDataType.FLOAT,
                    unit="degreesF",
                ),
                ChannelConfig(
                    name="dewP",
                    data_type=ChannelDataType.FLOAT,
                    unit="degreesF",
                ),
                ChannelConfig(
                    name="wdsp",
                    data_type=ChannelDataType.FLOAT,
                    unit="mph",
                ),
                ChannelConfig(
                    name="prcp",
                    data_type=ChannelDataType.FLOAT,
                    unit="in",
                ),
            ],
        ),
    ],
)

In [8]:
data_source = meteorological_data_source(file_path)

credentials: SiftChannelConfig = {
    "apikey": "1CTVOwfktFMNlWvctYte3jKoA61ub0S0AlDKfaQg",
    "uri": "http://localhost:50051",
    "use_ssl": False,
}
 
with use_sift_channel(credentials) as grpc_channel:
    # Create an ingestion service
    ingestion_service = IngestionService(grpc_channel, config)
 
    # Create a run to use during this period of ingestion
    run_name = f"[{ASSET_NAME}].{time.time():.0f}"
    ingestion_service.attach_run(grpc_channel, run_name)
    print('channel created')

    # Stream sensor data to Sift
    for row in data_source:
        print(row)
    
        channel_values = [
            float_value(row['temp']),
            float_value(row['dewp']),
            float_value(row['wdsp']),
            float_value(row['prcp'])
        ]

        ingestion_service.ingest_flows({
            "flow_name": "denver_weather",
            "timestamp": row['timestamp'],
            "channel_values": channel_values
        })

channel created
{'timestamp': datetime.datetime(2018, 1, 1, 7, 0, tzinfo=datetime.timezone.utc), 'temp': 11.6, 'dewp': 5.5, 'wdsp': 9.9, 'prcp': 0.0}
{'timestamp': datetime.datetime(2018, 1, 2, 7, 0, tzinfo=datetime.timezone.utc), 'temp': 21.2, 'dewp': 7.3, 'wdsp': 9.9, 'prcp': 0.0}
{'timestamp': datetime.datetime(2018, 1, 3, 7, 0, tzinfo=datetime.timezone.utc), 'temp': 31.8, 'dewp': 3.0, 'wdsp': 15.9, 'prcp': 0.0}
{'timestamp': datetime.datetime(2018, 1, 4, 7, 0, tzinfo=datetime.timezone.utc), 'temp': 34.6, 'dewp': 11.6, 'wdsp': 8.9, 'prcp': 0.0}
{'timestamp': datetime.datetime(2018, 1, 5, 7, 0, tzinfo=datetime.timezone.utc), 'temp': 36.3, 'dewp': 11.4, 'wdsp': 14.0, 'prcp': 0.0}
{'timestamp': datetime.datetime(2018, 1, 6, 7, 0, tzinfo=datetime.timezone.utc), 'temp': 40.2, 'dewp': 18.1, 'wdsp': 11.1, 'prcp': 0.0}
{'timestamp': datetime.datetime(2018, 1, 7, 7, 0, tzinfo=datetime.timezone.utc), 'temp': 39.8, 'dewp': 25.2, 'wdsp': 11.1, 'prcp': 0.01}
{'timestamp': datetime.datetime(2018,